# 06 – Temiz Veriyle Yeniden Eğitim ve Ölçüm

5. haftada veri setlerinde iki sorun bulundu:
- **Yinelenen satırlar:** Aynı cevap aynı soruda birden fazla kez geçiyordu.
- **Sızıntı:** "Görülmemiş sorular" test kümesindeki bazı sorular eğitim kümesinde de vardı; bazı test cevapları da eğitimdeki cevapların birebir aynısıydı.

Sızıntı, sınavdan önce soruların bir kısmını görmüş olmak gibidir: Model o sorularda "ezberden" iyi sonuç verir ve başarısı olduğundan yüksek görünür. Bu yüzden 05 numaralı not defterindeki sonuçların **temiz veriyle yeniden ölçülmesi** gerekiyor.

Bu not defterinde:
1. Temizlik kurallarını uygulayıp temiz veri tablosunu oluşturacağız.
2. İnce ayarlı modeli temiz veriyle yeniden eğitip ölçeceğiz.
3. Her tahmin için modelin **ne kadar emin olduğunu** gösteren bir ölçü (belirsizlik) hesaplayacağız. Bu ölçü 07 numaralı not defterinde güven skoru (FR-04) için kullanılacak.
4. Sonuçları 05'teki (temizlenmemiş veri) sonuçlarla karşılaştıracağız.

### Nasıl çalıştırılır?
1. **Önce GitHub'daki son değişikliklerin ana dala birleştirilmiş olması gerekir** (temizlik kodu `src/veri/temizle.py` orada).
2. Dosyayı Drive'daki `CevapAnaliz` klasörüne yükle ve Colab'da aç.
3. **GPU şart:** Çalışma zamanı → Çalışma zamanı türünü değiştir → **T4 GPU**.
4. **Çalışma zamanı → Tümünü çalıştır.** Toplam süre yaklaşık **20 dakika**; sekmeyi kapatma.

## 1. Kurulum

In [ ]:
import os, sys, subprocess
from pathlib import Path

COLAB = "google.colab" in sys.modules
if COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/CevapAnaliz")
    PROJE = Path("/content/CevapAnaliz")
    if PROJE.exists():
        subprocess.run(["git", "-C", str(PROJE), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/Semantik-Muhendisleri/CevapAnaliz.git", str(PROJE)], check=True)
    (DRIVE / "data").mkdir(parents=True, exist_ok=True)
    if not (PROJE / "data").exists():
        os.symlink(DRIVE / "data", PROJE / "data")
else:  # Kendi bilgisayarında çalıştırılırsa
    PROJE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    DRIVE = PROJE

os.chdir(PROJE)
sys.path.insert(0, str(PROJE))
SEKIL = DRIVE / "sekiller"; SEKIL.mkdir(parents=True, exist_ok=True)
SONUC = DRIVE / "sonuclar"; SONUC.mkdir(parents=True, exist_ok=True)

import torch
import pandas as pd
HIZLI_TEST = os.environ.get("CEVAPANALIZ_HIZLI_TEST") == "1"   # yalnızca geliştirici testi için
CIHAZ = "cuda" if torch.cuda.is_available() else "cpu"
if CIHAZ == "cpu" and not HIZLI_TEST:
    raise RuntimeError("GPU kapalı! Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU seçip "
                       "not defterini baştan çalıştır.")
if not (PROJE / "src" / "veri" / "temizle.py").exists():
    raise RuntimeError("Temizlik kodu bulunamadı. GitHub'daki pull request'i ana dala birleştirip "
                       "not defterini baştan çalıştır.")
print("GPU:", torch.cuda.get_device_name(0) if CIHAZ == "cuda" else "yok (hızlı test)")

## 2. Veri temizliği

Dört kural sırayla uygulanıyor:

| Kural | Ne yapıyor? | Neden? |
|---|---|---|
| **Çelişkili etiket** | Aynı soruya verilmiş aynı cevap farklı puanlarla geçiyorsa hepsini çıkarır | Hangi puanın doğru olduğunu bilemeyiz; model çelişkili örnekten bir şey öğrenemez |
| **Yinelenen satır** | Aynı soruya verilmiş aynı cevap birden fazla kez geçiyorsa birini bırakır | Model aynı örneği tekrar tekrar görmesin; ölçümde de aynı cevap iki kez sayılmasın |
| **Eğitimle aynı test cevabı** | Test kümesindeki cevap, eğitimde birebir varsa çıkarır | Model bu cevabı eğitimde zaten görmüş; test "görülmemiş" olmaz |
| **Sızan soru** | "Görülmemiş sorular" testinde olup eğitimde de geçen soruları çıkarır | Bu sorular aslında görülmüş; test adının hakkını vermez |

In [ ]:
from src.veri.temizle import temizle

ham = pd.read_csv(PROJE / "data" / "processed" / "ortak_veri.csv", keep_default_na=False)
df, rapor = temizle(ham)
df.to_csv(PROJE / "data" / "processed" / "temiz_veri.csv", encoding="utf-8")   # satır numaraları korunur

display(pd.Series(rapor, name="satır sayısı").to_frame())
karsilastirma = pd.DataFrame({"önce": ham.groupby(["veri_seti", "bolum"]).size(),
                              "sonra": df.groupby(["veri_seti", "bolum"]).size()})
karsilastirma["çıkan"] = karsilastirma["önce"] - karsilastirma["sonra"]
display(karsilastirma)

## 3. Test düzeni

Test düzeni 05 ile aynı, tek bir farkla:
- **Mohler:** Sorular 5 gruba ayrılıyor; her seferinde 4 grupla eğitilip kalan grupla test ediliyor (5 katlı çapraz doğrulama). 05'te bu ayırmayı kütüphane yapıyordu ve kütüphanenin farklı sürümleri soruları farklı gruplara dağıttığı için bizim bilgisayarımızdaki ve Colab'daki Mohler sonuçları biraz farklı çıkmıştı. **Artık sorular sabit bir kuralla** (soru numarası sırasına göre sırayla 1., 2., 3., 4., 5. gruba) dağıtılıyor; böylece sonuçlar her yerde aynı çıkar.
- **SciEntsBank ve Beetle:** Veri setinin kendi eğitim kümesiyle eğitilip görülmemiş cevaplar ve görülmemiş sorular kümeleriyle test ediliyor (artık temizlenmiş hâlleriyle).

In [ ]:
import numpy as np

def egitim_planlari():
    """(ad, eğitim satırları, {test adı: test satırları}, QWK basamağı) üretir."""
    mohler = df[df["veri_seti"] == "mohler"]
    sorular = sorted(mohler["soru_id"].unique())
    grup = {s: i % 5 for i, s in enumerate(sorular)}          # sabit kural: sırayla 5 gruba
    kat = mohler["soru_id"].map(grup)
    for k in range(5):
        yield (f"Mohler (kat {k + 1}/5)", mohler.index[kat != k].values, {"Mohler": mohler.index[kat == k].values}, 5)
    for veri_seti, ad in [("scientsbank", "SciEntsBank"), ("beetle", "Beetle")]:
        secili = df["veri_seti"] == veri_seti
        testler = {f"{ad} – görülmemiş cevaplar": df.index[secili & (df["bolum"] == "test_ua")].values,
                   f"{ad} – görülmemiş sorular": df.index[secili & (df["bolum"] == "test_uq")].values}
        yield ad, df.index[secili & (df["bolum"] == "train")].values, testler, 2

PLANLAR = list(egitim_planlari())
if HIZLI_TEST:   # geliştirici testi: çok küçük parçalar
    PLANLAR = [(a, e[:256], {t: s[:96] for t, s in ts.items()}, q) for a, e, ts, q in PLANLAR]
QWK_SEVIYE = {t: q for _, _, ts, q in PLANLAR for t in ts}
TEST_SIRA = ["Mohler", "SciEntsBank – görülmemiş cevaplar", "SciEntsBank – görülmemiş sorular",
             "Beetle – görülmemiş cevaplar", "Beetle – görülmemiş sorular"]
for ad, egitim, testler, _ in PLANLAR:
    print(f"{ad:<22} eğitim: {len(egitim):>5}   test: " + ", ".join(f"{t.split(' – ')[-1]} {len(s)}" for t, s in testler.items()))

## 4. Model, eğitim ve belirsizlik

**Model ve eğitim ayarları 05 ile aynı:** `cross-encoder/stsb-distilroberta-base` çapraz kodlayıcısı, referans ve öğrenci cevabını birlikte okuyup 0–1 arası puan verecek şekilde öğretmen puanlarıyla 3 epok eğitiliyor (yığın 16, öğrenme oranı 0,00002).

**Yeni: Belirsizlik (modelin ne kadar emin olduğu).** Dil modellerinin içinde **dropout** adı verilen bir mekanizma vardır: Eğitim sırasında her adımda nöronların bir kısmı rastgele kapatılır; böylece model tek bir nörona fazla güvenmemeyi öğrenir. Normalde puan verirken dropout kapatılır.

Biz puan verirken dropout'u **bilerek açıp** aynı cevabı modele **10 kez** puanlatıyoruz. Her seferinde farklı nöronlar kapandığı için puanlar biraz değişir:
- Puanlar birbirine çok yakınsa (örneğin 0,81 · 0,83 · 0,80 …) model **emin** demektir.
- Puanlar birbirinden çok farklıysa (örneğin 0,35 · 0,72 · 0,51 …) model **emin değil** demektir.

10 puanın standart sapmasına **belirsizlik** diyoruz. Bu yönteme literatürde **Monte Carlo dropout** denir. Asıl puan olarak yine dropout kapalıyken verilen puanı kullanıyoruz.

In [ ]:
import time
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

MODEL_ADI = "cross-encoder/stsb-distilroberta-base"
AYAR = {"epok": 3, "batch": 16, "ogrenme_orani": 2e-5, "max_uzunluk": 160, "mc_tur": 10}
if HIZLI_TEST:
    MODEL_ADI, AYAR["epok"], AYAR["mc_tur"] = "cross-encoder/stsb-TinyBERT-L4", 1, 3

tokenizer = AutoTokenizer.from_pretrained(MODEL_ADI)
GPU = CIHAZ == "cuda"

def kodla(satirlar):
    return tokenizer(list(df.loc[satirlar, "referans_cevap"]), list(df.loc[satirlar, "ogrenci_cevap"]),
                     truncation=True, max_length=AYAR["max_uzunluk"], padding=True, return_tensors="pt").to(CIHAZ)

@torch.no_grad()
def puanla(model, satirlar):
    sonuc = []
    for i in range(0, len(satirlar), 128):
        with torch.autocast(device_type=CIHAZ, enabled=GPU):
            cikti = model(**kodla(satirlar[i:i + 128])).logits.squeeze(-1)
        sonuc.append(torch.sigmoid(cikti.float()).cpu().numpy())
    return np.concatenate(sonuc)

def tahmin_ve_belirsizlik(model, satirlar):
    """Dropout kapalıyken verilen puanı ve dropout açıkken verilen 10 puanın standart sapmasını döndürür."""
    model.eval()
    tahmin = puanla(model, satirlar)
    model.train()                       # dropout açık
    torch.manual_seed(0)
    turlar = np.stack([puanla(model, satirlar) for _ in range(AYAR["mc_tur"])])
    model.eval()
    return tahmin, turlar.std(axis=0)

def egit(egitim_satirlari, tohum=0):
    torch.manual_seed(tohum)
    rastgele = np.random.default_rng(tohum)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_ADI).to(CIHAZ)
    optimizer = torch.optim.AdamW(model.parameters(), lr=AYAR["ogrenme_orani"], weight_decay=0.01)
    adim = AYAR["epok"] * int(np.ceil(len(egitim_satirlari) / AYAR["batch"]))
    zamanlayici = get_linear_schedule_with_warmup(optimizer, int(0.1 * adim), adim)
    olcekleyici = torch.amp.GradScaler(enabled=GPU)
    kayip_fonksiyonu = torch.nn.BCEWithLogitsLoss()
    for epok in range(AYAR["epok"]):
        model.train()
        sira, toplam = rastgele.permutation(egitim_satirlari), 0.0
        for i in range(0, len(sira), AYAR["batch"]):
            yigin = sira[i:i + AYAR["batch"]]
            puan = torch.tensor(df.loc[yigin, "puan"].values, dtype=torch.float32, device=CIHAZ)
            with torch.autocast(device_type=CIHAZ, enabled=GPU):
                kayip = kayip_fonksiyonu(model(**kodla(yigin)).logits.squeeze(-1).float(), puan)
            optimizer.zero_grad()
            olcekleyici.scale(kayip).backward()
            olcekleyici.step(optimizer)
            olcekleyici.update()
            zamanlayici.step()
            toplam += kayip.item() * len(yigin)
        print(f"    epok {epok + 1}/{AYAR['epok']}: ortalama kayıp {toplam / len(sira):.4f}")
    return model

print("Model:", MODEL_ADI)

## 5. Eğitim

7 model sırayla eğitiliyor (Mohler için 5, SciEntsBank ve Beetle için birer). Her modelin eğitimde görmediği test cevaplarına verdiği puanlar ve belirsizlikler kaydediliyor. GPU ile yaklaşık 15–20 dakika sürer.

In [ ]:
kayitlar = []
toplam_baslangic = time.time()
for ad, egitim_satirlari, testler, _ in PLANLAR:
    baslangic = time.time()
    print(f"{ad}: {len(egitim_satirlari):,} cevapla eğitiliyor".replace(",", "."))
    model = egit(egitim_satirlari)
    for test_adi, test_satirlari in testler.items():
        tahmin, belirsizlik = tahmin_ve_belirsizlik(model, test_satirlari)
        kayitlar.append(pd.DataFrame({"test": test_adi, "tahmin": tahmin, "belirsizlik": belirsizlik},
                                     index=test_satirlari))
    del model
    if GPU:
        torch.cuda.empty_cache()
    print(f"    bitti ({time.time() - baslangic:.0f} saniye)\n")
print(f"Toplam süre: {(time.time() - toplam_baslangic) / 60:.1f} dakika")

tahminler = pd.concat(kayitlar).join(df[["veri_seti", "bolum", "soru_id", "orijinal_puan", "puan"]])
tahminler.to_csv(SONUC / "06_tahminler.csv", encoding="utf-8")
print("Kaydedildi:", SONUC / "06_tahminler.csv")

## 6. Sonuçlar: Temiz veri ile temizlenmemiş veri

Aynı model ve aynı ayarlarla, temizlenmemiş veride (05) ve temiz veride (06) elde edilen sonuçları karşılaştırıyoruz. Temiz verideki sonuç **gerçek başarıyı** gösterir.

In [ ]:
from src.puanlama.olcutler import olc

satirlar = []
for test_adi in TEST_SIRA:
    t = tahminler[tahminler["test"] == test_adi]
    satirlar.append({"test": test_adi, "yontem": "06 – temiz veri", "n": len(t),
                     **olc(t["puan"], t["tahmin"], QWK_SEVIYE[test_adi])})
sonuc = pd.DataFrame(satirlar)
sonuc.round(4).to_csv(SONUC / "06_temiz_veri.csv", index=False, encoding="utf-8")

eski_dosya = SONUC / "05_ince_ayar.csv"
if eski_dosya.exists() and not HIZLI_TEST:
    eski = pd.read_csv(eski_dosya)
    eski = eski[eski["yontem"] == "İnce ayardan sonra"].assign(yontem="05 – temizlenmemiş veri")
    sonuc = pd.concat([eski, sonuc], ignore_index=True)
YONTEM_SIRA = [y for y in ["05 – temizlenmemiş veri", "06 – temiz veri"] if y in set(sonuc["yontem"])]

for olcut in ["pearson", "qwk"]:
    print(f"\n{olcut.upper()}")
    display(sonuc.pivot(index="yontem", columns="test", values=olcut).loc[YONTEM_SIRA, TEST_SIRA].round(3))

In [ ]:
import matplotlib.pyplot as plt

RENK = {"05 – temizlenmemiş veri": "#2a78d6", "06 – temiz veri": "#eb6834"}
fig, eksenler = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
genislik = 0.8 / len(YONTEM_SIRA)
for ax, olcut, baslik in zip(eksenler, ["pearson", "qwk"], ["Pearson korelasyonu", "QWK"]):
    for i, yontem in enumerate(YONTEM_SIRA):
        degerler = sonuc[sonuc["yontem"] == yontem].set_index("test").loc[TEST_SIRA, olcut]
        x = np.arange(len(TEST_SIRA)) + (i - (len(YONTEM_SIRA) - 1) / 2) * genislik
        ax.bar(x, degerler, width=genislik - 0.03, color=RENK[yontem], label=yontem)
    ax.axhline(0.70, color="#52514e", linewidth=1, linestyle="--")
    ax.set_xticks(np.arange(len(TEST_SIRA)), [t.replace(" – ", "\n") for t in TEST_SIRA], fontsize=8.5)
    ax.set_title(baslik, loc="left", fontsize=12)
    ax.set_ylim(0, 1)
    for kenar in ["top", "right"]:
        ax.spines[kenar].set_visible(False)
    ax.grid(axis="y", color="#e6e5e0"); ax.set_axisbelow(True)
eksenler[0].text(len(TEST_SIRA) - 0.5, 0.71, "hedef 0,70", ha="right", va="bottom", fontsize=9, color="#52514e")
tutamaklar, etiketler = eksenler[0].get_legend_handles_labels()
fig.legend(tutamaklar, etiketler, frameon=False, loc="upper left", ncol=2, bbox_to_anchor=(0.01, 0.93), fontsize=10)
fig.suptitle("Temiz veriyle yeniden ölçüm", x=0.01, ha="left", fontsize=14, fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.88))
fig.savefig(SEKIL / "06_temiz_veri.png", dpi=150, bbox_inches="tight")
plt.show()

### Sonuçların yorumu
Aşağıdaki hücre yorumu tablodaki sayılara bakarak kendisi yazar.

In [ ]:
def virgul(x):
    return f"{x:.2f}".replace(".", ",")

tablo = {o: sonuc.pivot(index="yontem", columns="test", values=o) for o in ["pearson", "qwk"]}
for olcut, ad in [("pearson", "Pearson"), ("qwk", "QWK")]:
    print(f"── {ad} ──")
    for test_adi in TEST_SIRA:
        yeni = tablo[olcut].loc["06 – temiz veri", test_adi]
        satir = f"{test_adi}: temiz veri {virgul(yeni)}"
        if "05 – temizlenmemiş veri" in tablo[olcut].index:
            eski = tablo[olcut].loc["05 – temizlenmemiş veri", test_adi]
            satir += f" (temizlenmemiş veride {virgul(eski)}, fark {'+' if yeni >= eski else ''}{virgul(yeni - eski)})"
        satir += " | hedefe ulaşıldı ✓" if yeni >= 0.70 else f" | hedefe {virgul(0.70 - yeni)} kaldı"
        print("  " + satir)
    print()

**Tabloyu okurken:**
- Temiz verideki sonuçlar, temizlenmemiş verideki sonuçlardan **biraz düşükse** bu beklenen bir durumdur: Sızıntı başarıyı yapay olarak yükseltiyordu. Temiz verideki sonuç gerçek başarıdır.
- En büyük değişikliğin **Beetle – görülmemiş sorular** testinde olması beklenir, çünkü sızan 2 soru bu testteydi.
- Mohler'de fark, hem temizlikten hem de grupların artık sabit kuralla oluşturulmasından kaynaklanır.

## Özet

- Veri temizlendi; temiz tablo Drive'da `CevapAnaliz/data/processed/temiz_veri.csv` dosyasına kaydedildi.
- İnce ayarlı model temiz veriyle yeniden eğitildi ve ölçüldü; sonuçlar `sonuclar/06_temiz_veri.csv`, grafik `sekiller/06_temiz_veri.png`.
- Her test cevabı için puan ve **belirsizlik** `sonuclar/06_tahminler.csv` dosyasına kaydedildi.

**Sonraki not defteri: 07 – Güven Skoru, Eksik Kavram Analizi ve Girdi Kontrolü.** Belirsizliği kullanarak emin olunamayan cevapları öğretmene yönlendireceğiz (FR-04), öğrencinin eksik bıraktığı kavramları bulacağız (NFR-03) ve CSV girdi kontrolünü (FR-06) göstereceğiz.